# Online Retail Dataset
## Data Quality Assessment and Cleaning

#### Overview
This notebook performs initial data quality assessment and preprocessing of the Online Retail dataset to ensure analytical reliability prior to customer-level analysis.

#### Objective
To evaluate dataset completeness, consistency, and structural integrity, and apply appropriate cleaning procedures required for accurate Recency, Frequency, and Monetary (RFM) analysis.

#### Activities Covered
The notebook undertakes the following stages:
- Data Inspection – Examine dataset structure, data types, missing values, duplicates, and summary statistics.
- Data Quality Assessment – Identify inconsistencies, invalid transactions, and business logic violations affecting retail analytics.
- Data Cleaning –
Handle missing values
Remove invalid transactional records
Correct data types
Preserve legitimate retail transaction structure
Control extreme values through outlier treatment
- Dataset Preparation – Produce a clean, analysis-ready dataset suitable for downstream exploratory analysis and customer segmentation.

The cleaned dataset is exported for use in subsequent notebooks dedicated to Exploratory Data Analysis (EDA), Feature Engineering, and RFM-based Customer Segmentation.

In [27]:
#import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Initial Data Quality Assessment

#### Objectives
To assess the Online Retail dataset for completeness and integrity to guide cleaning for accurate RFM customer segmentation.

We examine:
- Data Preview – To understand dataset structure, variable layout, and formatting by inspecting the first few rows and overall dataset shape.
- Data Types & Missing Values – To assess variable types, detect missing data, and determine fields requiring type conversion to enable correct aggregation and analysis.
- Duplicate Checks – To identify repeated rows, invoices, or product entries using full-row and subset duplication checks, ensuring dataset integrity and preventing inflation of transactional metrics.
- Summary Statistics – To understand numeric variable behavior, including central tendency, spread, and extremes, and detect potential anomalies such as negative quantities or extreme pricing.
- Outlier Detection – To identify extreme values using percentile-based and IQR methods, ensuring that unusual transactions are flagged and their impact on frequency, monetary, and revenue analysis is understood.
- Business Logic Validation – To verify that numeric values align with expected operational rules, including:
o	Missing CustomerID
o	Zero or Negative Quantity and UnitPrice
o	Duplicate invoices or stock entries
 These checks confirm operational consistency and highlight records requiring corrective action.

#### Key Findings
1.	**Dataset Size and Structure:** - The dataset contains 541,909 records and 8 columns, comprising 1 integer variable (Quantity), 2 float variables (UnitPrice, CustomerID), and 5 object variables (InvoiceNo, StockCode, Description, InvoiceDate, Country). The dataset is generally complete, with manageable memory usage (~33.1 MB).
2. **Missing Values:** - Missingness is observed in Description (1,454 records) and CustomerID (135,080 records, ~25%), which may impact customer-level aggregation and product-level analysis.
3.	**Data Types:**
The InvoiceDate column is stored as an object rather than datetime, and CustomerID is stored as a float. Both require type conversion to enable time-based analysis and proper customer-level grouping.
4.	**Duplicate Records** – There are 5,268 exact duplicate rows (~0.97%) that need removal, while there are 516,009 duplicate InvoiceNo entries and 537,839 duplicate StockCode entries, reflecting legitimate multi-line invoices and repeated product sales that should be preserved.
5.	**Descriptive Statistics Analysis** – Quantity values range from -80,995 to 80,995 (mean 9.55, median 3), UnitPrice ranges from -11,062.06 to 38,970.00 (mean 4.61, median 2.08), and CustomerID ranges from 12,346 to 18,287 (mean 15,287.69) with 135,080 missing records; extreme Quantity and UnitPrice values indicate returns, bulk purchases, or anomalous pricing entries, while missing CustomerIDs highlight gaps that could affect customer-level aggregation and RFM analysis.
8.	**Outlier Analysis** – Extreme Quantity values total 58,619 records (10.82%) and extreme UnitPrice values total 39,627 records (7.31%); these outliers reflect returns, bulk purchases, or anomalous pricing entries that could disproportionately affect Frequency and Monetary calculations if not addressed.
9.	**Business Logic Validation** – Negative Quantity (10,624 transactions) represents returns or adjustments, while negative UnitPrice (2517 transactions) likely reflects rare errors or corrections. Proper handling of these values is necessary to ensure valid Recency, Frequency, and Monetary calculations.


In [28]:
# load the data
df = pd.read_csv('../data/data_raw/online_retail.csv')

# inspect the data
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [29]:
# i) Check for data type and missing data
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [30]:
# ii) Check for typos and extra spaces in column names
df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='object')

In [31]:
# standardize column names by removing extra spaces
df.columns=df.columns.str.strip()

In [32]:
# iii) Check for duplicate columns
df.columns.duplicated()

array([False, False, False, False, False, False, False, False])

In [33]:
# iv) check for duplicated rows
df.duplicated().sum()

np.int64(5268)

In [34]:
# v) inspect summary statistics
df.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


In [35]:
# vi) Check for Outliers
# Detect outliers using IQR method for all numeric columns
numeric_cols = ['Quantity','UnitPrice']
outlier_summary = []

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[
        (df[col] < lower_bound) | 
        (df[col] > upper_bound)
    ]
    
    outlier_count = outliers.shape[0]
    outlier_percentage = (outlier_count / len(df)) * 100
    
    outlier_summary.append({
        "Column": col,
        "Outlier Count": outlier_count,
        "Outlier %": round(outlier_percentage, 2)
    })

outlier_summary_df = pd.DataFrame(outlier_summary)
outlier_summary_df

,Column,Outlier Count,Outlier %
0,Quantity,58619,10.82
1,UnitPrice,39627,7.31


In [36]:
# vii) Check business logic
validation_results = {
    "missing_customerID": df['CustomerID'].isna().sum(),
    "duplicate_invoice": df.duplicated(subset=['InvoiceNo']).sum(),
    "duplicate_stock": df.duplicated(subset=['StockCode']).sum(),
    "zero_or_negative_quantity": (df['Quantity'] <= 0).sum(),
    "zero_or_negative_price": (df['UnitPrice'] <= 0).sum()
}

validation_results

{'missing_customerID': np.int64(135080),
 'duplicate_invoice': np.int64(516009),
 'duplicate_stock': np.int64(537839),
 'zero_or_negative_quantity': np.int64(10624),
 'zero_or_negative_price': np.int64(2517)}

## Data Cleaning

#### Objective
To clean and preprocess the Online Retail dataset for accurate RFM customer segmentation.

We will:
- Handle Missing Data – Remove transactions with missing CustomerID and replace missing Description values with “Unknown.”
- Validate and Correct Numeric Fields – Remove transactions with zero or negative Quantity and UnitPrice.
- Remove Exact Duplicates – Eliminate fully duplicated rows while preserving legitimate multi-line invoices and repeated product entries.
- Correct Datatypes – Convert identifiers (InvoiceNo, StockCode, CustomerID) to string and InvoiceDate to datetime.
- Handle Outliers – Cap extreme Quantity and UnitPrice values at the 99th percentile to control anomalous transactions.

#### Cleaning Actions
1.	Transactions with missing CustomerID (135,080 records, ~25% of total) were removed because RFM analysis requires identifiable customers to compute Recency, Frequency, and Monetary metrics, and retaining unassigned transactions would distort customer-level aggregation and compromise segmentation validity.
2.	Missing Description values were replaced with “Unknown” because they reflect incomplete product-level information rather than missing transaction data, and since RFM metrics are computed at the customer level and do not depend on product descriptions, filling them preserves valid purchase records without compromising analytical accuracy.
3.	Transactions with zero or negative quantities were removed because they represent returns or adjustments rather than actual purchases, and including them would distort Frequency and Monetary calculations, so excluding them ensures that Recency, Frequency, and Monetary metrics accurately reflect true purchase behavior for reliable customer segmentation.
4.	Transactions with zero or negative UnitPrice were removed because they likely represent data-entry errors or system adjustments, and including them would distort customer revenue calculations, so excluding these invalid records ensures that the Monetary metric accurately reflects true spending and preserves the integrity of RFM-based segmentation.
5.	Exact duplicate rows were removed to prevent inflated purchase frequency and overstated customer revenue in RFM calculations, ensuring each transaction is counted only once while retaining legitimate multiple product lines under the same invoice to preserve analytical integrity.
6.	Duplicate InvoiceNo and StockCode entries were preserved because they reflect the normal retail structure of multi-line invoices and repeated product sales across customers, and removing them would eliminate valid transaction lines and distort Frequency and Monetary calculations, whereas only exact full-row duplicates were removed to maintain accurate RFM aggregation.
7.	Identifier fields like InvoiceNo, StockCode, and CustomerID were converted to string type because they represent unique labels rather than numeric values, and storing them as strings preserves their categorical meaning, prevents unintended calculations, and ensures accurate grouping and customer-level analysis.
8.	InvoiceDate was converted to datetime because RFM analysis requires accurate calculation of Recency as the difference between a reference date and the last purchase date, and using a proper datetime format ensures correct time-based computations, sorting, grouping, and supports reliable temporal analyses.
9.	Percentiles were examined before handling outliers to identify extreme Quantity and UnitPrice values that deviate disproportionately from the typical distribution, providing statistical justification for controlling only anomalous tail values while preserving the majority of legitimate transactions for accurate RFM segmentation.
10.	Quantity and UnitPrice were capped at the 99th percentile to control extreme upper-tail values that could disproportionately inflate Monetary calculations, ensuring that Recency, Frequency, and Monetary metrics reflect typical purchasing behavior while preserving 99% of genuine transaction data for accurate and stable RFM segmentation.
11.	The Country column was validated to ensure consistent labeling, correct formatting, and absence of duplicates or nulls, confirming that all 37 unique countries are structurally consistent and analytically reliable, so no cleaning or transformation was needed for accurate RFM analysis.
12.	Duplicate occurrences of InvoiceNo, StockCode, and CustomerID reflect legitimate transactional behavior—multi-product invoices, repeated product sales, and returning customers—so these identifier fields do not require additional validation, unlike categorical variables such as Country, which need checking for classification accuracy.
13.	TotalPrice was calculated as Quantity × UnitPrice after capping outliers to ensure that transaction-level revenue reflects realistic values, preventing extreme transactions from inflating Monetary metrics and guaranteeing stable, reliable customer segmentation.

#### Status
- No missing CustomerID; negative or zero Quantity/UnitPrice removed, outliers capped.
- InvoiceNo, StockCode, CustomerID as strings; InvoiceDate as datetime; TotalPrice computed.
- Categorical fields consistent; legitimate duplicates preserved.
- Cleaned dataset saved as cleaned_online_retail.csv in the data/processed_data subfolder for downstream EDA and RFM analysis.


In [37]:
#Handle Missing Data
# i) Remove transactions with missing customerID for customer-level analysis
df_clean = df.dropna(subset = ['CustomerID']).copy()
df_clean['CustomerID'].isna().sum()


np.int64(0)

In [38]:
# ii) replace miising values in Description with 'Unknown'
df_clean['Description'] = df_clean['Description'].fillna('Unknown')
df_clean['Description'].isna().sum()

np.int64(0)

In [39]:
# iii) Remove transactions with zero or negative quantities
df_clean = df_clean[df_clean['Quantity'] > 0].copy()
(df_clean['Quantity'] <= 0).sum()

np.int64(0)

In [40]:
# iv) Remove transactions with zero or negative unitprice
df_clean = df_clean[df_clean['UnitPrice'] > 0].copy()
(df_clean['UnitPrice'] <= 0).sum()

np.int64(0)

In [41]:
# v) Remove exact duplicate rows
df_clean = df_clean.drop_duplicates()
df_clean.duplicated().sum()

np.int64(0)

In [42]:
# vi) Correct datatype
# a) Convert all identifiers to string type
df_clean['CustomerID'] = df_clean['CustomerID'].astype('str')
df_clean['InvoiceNo'] = df_clean['InvoiceNo'].astype('str')
df_clean['StockCode'] = df_clean['StockCode'].astype('str')


In [43]:
# b) Convert InvoiceDate to datetime
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate']) 

#check date business logic
(df_clean['InvoiceDate'] > pd.Timestamp.today()).sum()

np.int64(0)

In [44]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 392692 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    392692 non-null  object        
 1   StockCode    392692 non-null  object        
 2   Description  392692 non-null  object        
 3   Quantity     392692 non-null  int64         
 4   InvoiceDate  392692 non-null  datetime64[ns]
 5   UnitPrice    392692 non-null  float64       
 6   CustomerID   392692 non-null  object        
 7   Country      392692 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(5)
memory usage: 27.0+ MB


In [45]:
# vii) Handle outliers in Quantity and UnitPrice
# a) Check for extreme values in Quantity and UnitPrice
df_clean[['Quantity','UnitPrice']].describe(percentiles=[0.90,0.95,0.99])

,Quantity,UnitPrice
count,392692.000000,392692.000000
mean,13.119702,3.125914
std,180.492832,22.241836
min,1.000000,0.001000
50%,6.000000,1.950000
90%,24.000000,6.350000
95%,36.000000,8.500000
99%,120.000000,14.950000
max,80995.000000,8142.750000


In [46]:
# b) Cap Quantity at 99th percentile
quantity_cap = df_clean['Quantity'].quantile(0.99)
df_clean['Quantity'] = df_clean['Quantity'].clip(upper=quantity_cap)

In [47]:
# c) Cap UnitPrice at 99th percentile
price_cap = df_clean['UnitPrice'].quantile(0.99)
df_clean['UnitPrice'] = df_clean['UnitPrice'].clip(upper=price_cap)

In [48]:
# viii) validate categorical variable Country 
df_clean['Country'].nunique()

37

In [49]:
df_clean['Country'].value_counts()

Country
United Kingdom          349203
Germany                   9025
France                    8326
EIRE                      7226
Spain                     2479
Netherlands               2359
Belgium                   2031
Switzerland               1841
Portugal                  1453
Australia                 1181
Norway                    1071
Italy                      758
Channel Islands            747
Finland                    685
Cyprus                     603
Sweden                     450
Austria                    398
Denmark                    380
Poland                     330
Japan                      321
Israel                     245
Unspecified                241
Singapore                  222
Iceland                    182
USA                        179
Canada                     151
Greece                     145
Malta                      112
United Arab Emirates        68
European Community          60
RSA                         57
Lebanon                     45


In [50]:
# ix) Create TotalPrice column
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['UnitPrice']

In [51]:
# Check Data status in readiness for RFM 
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 392692 entries, 0 to 541908
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    392692 non-null  object        
 1   StockCode    392692 non-null  object        
 2   Description  392692 non-null  object        
 3   Quantity     392692 non-null  int64         
 4   InvoiceDate  392692 non-null  datetime64[ns]
 5   UnitPrice    392692 non-null  float64       
 6   CustomerID   392692 non-null  object        
 7   Country      392692 non-null  object        
 8   TotalPrice   392692 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(5)
memory usage: 30.0+ MB


In [52]:
df_clean.to_csv('../data/data_processed/cleaned_online_retail.csv',index=False)